# colab_14 — Geneformer continued pretraining (CPT), **per-study regime** · N=1 pilot

The third CPT regime in the locked design (after zero-shot and the aggregated regime in colab_11).
Instead of one CPT run on the concatenated + shuffled train split, this trains **three independent
LoRA adapters from the same frozen base** — one per study (SEA-AD, Li2025, Haney2024) — each on that
study's slice of the *same frozen donor split*. Parallel/independent, not sequential: no adapter sees
another study's gradients.

Each checkpoint is then re-embedded over the full glia substrate and passed through detector #1
(drift vs the colab_09 zero-shot baseline), exactly as the aggregated run was. Evals #1/#2/#3 on the
three checkpoints are a **separate downstream notebook** (mirrors how colab_11 was training-only and
colab_12/13 carried the evals).

**Outputs:** three LoRA adapters, three L−1 embeddings over the full substrate, and a
`geneformer_cpt_per_study` block in `audit_report.json` with per-study training budget + detector #1.

## 1 — Setup

### 1a — Mount Drive, clone/pull the repo, install the Geneformer environment, set the SMOKE switch

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info >= (3, 10), f"Geneformer needs Python >=3.10, got {sys.version_info[:2]}."

!pip install -r {REPO_PATH}/requirements_geneformer.txt

# Pin the Geneformer clone to the commit every prior FM notebook used, and assert HEAD
# (a moving HF main silently changes the tokenizer / model code -- see colab_13's crash arc).
GENEFORMER_REPO   = "/content/Geneformer"
GENEFORMER_COMMIT = "04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5"
if not os.path.exists(GENEFORMER_REPO):
    !git lfs install
    !git clone https://huggingface.co/ctheodoris/Geneformer {GENEFORMER_REPO}
    !git -C {GENEFORMER_REPO} checkout {GENEFORMER_COMMIT}
_head = subprocess.run(["git", "-C", GENEFORMER_REPO, "rev-parse", "HEAD"],
                       capture_output=True, text=True).stdout.strip()
assert _head == GENEFORMER_COMMIT, f"Geneformer HEAD {_head} != pinned {GENEFORMER_COMMIT}"
!cd {GENEFORMER_REPO} && pip install .
# torchao (Colab-preinstalled, unused) hard-raises inside peft dispatch below its floor -- remove it.
!pip uninstall -y torchao -q

import torch
assert torch.cuda.is_available(), "no CUDA GPU -- select a GPU runtime before CPT"
print("Python:", sys.version.split()[0], "| Geneformer commit:", GENEFORMER_COMMIT,
      "| GPU:", torch.cuda.get_device_name(0))

# --- the one live switch; ALL variable output paths derive from SUFFIX ---
SMOKE           = False
SMOKE_N_PER_GROUP = 40      # cells kept per (lineage x split x substate) group when SMOKE
SMOKE_MAX_STEPS = 8         # tiny fixed budget per study when SMOKE (plumbing only)
SUFFIX          = "_SMOKE" if SMOKE else ""
SEED            = 0
from datetime import date
TODAY           = date.today().isoformat()
print(f"SMOKE={SMOKE} | output suffix='{SUFFIX}'")

### 1b — pip freeze + env JSON (records the exact CPT-run stack)

In [ ]:
import json, platform, subprocess, sys

NOTEBOOK_ID = "colab_14"
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return None

env_snapshot = {
    "notebook_id":    NOTEBOOK_ID,
    "date":           TODAY,
    "python_version": sys.version,
    "platform":       platform.platform(),
    "os_release":     platform.release(),
    "gpu":            _run(["nvidia-smi", "-L"]),
    "cuda":           _run(["nvcc", "--version"]),
    "git_commit":     _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "geneformer_commit":    GENEFORMER_COMMIT,
    "scanpy_version":       _ver("scanpy"),
    "anndata_version":      _ver("anndata"),
    "torch_version":        _ver("torch"),
    "transformers_version": _ver("transformers"),
    "peft_version":         _ver("peft"),
    "accelerate_version":   _ver("accelerate"),
    "datasets_version":     _ver("datasets"),
    "geneformer_version":   _ver("geneformer"),
}
ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

## 2 — Load the substrate and validate the schema

### 2a — Rebuild the glia substrate (deterministic; same `cell_index` as every prior FM notebook)

In [ ]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp
sc.settings.verbosity = 1

MICRO_PATH = os.path.join(DRIVE_ROOT, "micro_subset", "micro_subset.h5ad")
ASTRO_PATH = os.path.join(DRIVE_ROOT, "astro_subset", "astro_subset.h5ad")
for p in (MICRO_PATH, ASTRO_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing labelled subset {p} (colab_07 / colab_08 output)")

micro = sc.read_h5ad(MICRO_PATH)
astro = sc.read_h5ad(ASTRO_PATH)
assert list(micro.var_names) == list(astro.var_names), "gene panels differ between subsets"
micro.obs["lineage"] = "microglia"
astro.obs["lineage"] = "astrocyte"
KEEP_OBS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id", "total_counts"]
micro.obs = micro.obs[[c for c in KEEP_OBS if c in micro.obs.columns]].copy()
astro.obs = astro.obs[[c for c in KEEP_OBS if c in astro.obs.columns]].copy()
glia = ad.concat([micro, astro], join="inner", index_unique="-")
del micro, astro; gc.collect()
glia.obs["cell_index"] = np.arange(glia.n_obs)
print("combined glia:", glia.shape)
print("lineage:", glia.obs["lineage"].value_counts().to_dict())
print("study_id:", glia.obs["study_id"].value_counts().to_dict())

# raw-counts guard -- Geneformer rank-encodes RAW counts.
_idx = np.random.default_rng(0).choice(glia.n_obs, size=min(2000, glia.n_obs), replace=False)
Xs = glia.X[_idx]
data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f})"
assert glia.n_obs == 142588, f"expected 142,588-cell substrate, got {glia.n_obs} -- subset files changed"

### 2b — Fail loud on the obs schema this notebook depends on

In [ ]:
# Per-study CPT + detector #1 need lineage/substate/apoe/study/donor complete; `region` is NOT
# needed here (it was an eval-#2 confound-audit column in colab_12), so it is deliberately not required.
REQUIRED_OBS = ["cell_index", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]
missing_cols = [c for c in REQUIRED_OBS if c not in glia.obs.columns]
assert not missing_cols, f"substrate missing required obs columns: {missing_cols}"

for col in REQUIRED_OBS:
    n_null = int(pd.isna(glia.obs[col]).sum())
    assert n_null == 0, f"{col} has {n_null} null values -- CPT/detector audits require it complete"

assert set(glia.obs["lineage"]) == {"microglia", "astrocyte"}, "unexpected lineage values"
assert set(glia.obs["substate"]) <= {"homeostatic", "activated", "resting", "reactive", "intermediate"}, \
    f"unexpected substate values: {set(glia.obs['substate'])}"
assert set(glia.obs["apoe_carrier"]) <= {"carrier", "noncarrier", "e2"}, \
    f"unexpected apoe_carrier values: {set(glia.obs['apoe_carrier'])}"

STUDIES = ["SEA-AD", "Li2025", "Haney2024"]
assert set(glia.obs["study_id"]) == set(STUDIES), \
    f"study_id values {set(glia.obs['study_id'])} != expected {set(STUDIES)}"
print("obs schema OK | studies:", STUDIES)

## 3 — Held-out split verification

### 3a — Rebuild the frozen donor split, hard-stop on any mismatch, then per-study train counts

In [ ]:
import json
from sklearn.model_selection import train_test_split

DONOR_META_PATH = os.path.join(REPO_PATH, "outputs", "donor_metadata.csv")
SPLIT_FRACS     = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED_POOL       = range(200)

assert os.path.exists(DONOR_META_PATH), f"need committed donor metadata {DONOR_META_PATH} (colab_11 minted it)"
donor_meta = pd.read_csv(DONOR_META_PATH, dtype=str)
sub_donors = set(glia.obs["donor_id"].astype(str))
donor_meta = donor_meta[donor_meta["donor_id"].isin(sub_donors)].reset_index(drop=True)
assert donor_meta["donor_id"].is_unique, "a donor_id maps to >1 metadata row"

def _study_split(seed):
    d_tr, d_te = train_test_split(donor_meta["donor_id"], test_size=SPLIT_FRACS["test"],
                                  random_state=seed, stratify=donor_meta["study_id"])
    strat_tr = donor_meta.set_index("donor_id").loc[d_tr.values, "study_id"]
    d_tr, d_va = train_test_split(d_tr, test_size=SPLIT_FRACS["val"] / (SPLIT_FRACS["train"] + SPLIT_FRACS["val"]),
                                  random_state=seed, stratify=strat_tr)
    return set(d_tr), set(d_va), set(d_te)

def _test_margin(d_test):
    m = donor_meta[donor_meta["donor_id"].isin(d_test)]
    return int(min((m["apoe_carrier"] == "carrier").sum(), (m["apoe_carrier"] == "noncarrier").sum(),
                   (m["diagnosis"] == "AD").sum(), (m["diagnosis"] == "Control").sum()))

best_seed = max(SEED_POOL, key=lambda s: _test_margin(_study_split(s)[2]))
d_train, d_val, d_test = _study_split(best_seed)
margin = _test_margin(d_test)
split_map = {**{d: "train" for d in d_train}, **{d: "val" for d in d_val}, **{d: "test" for d in d_test}}
glia.obs["split"] = glia.obs["donor_id"].astype(str).map(split_map).astype("category")
assert not glia.obs["split"].isna().any(), "some cells' donor received no split assignment"

n_donors = {k: int(sum(1 for v in split_map.values() if v == k)) for k in ("train", "val", "test")}
n_cells  = glia.obs["split"].value_counts().to_dict()
test_by_study = glia.obs.loc[glia.obs["split"] == "test", "study_id"].value_counts(normalize=True).round(3).to_dict()
print(f"seed {best_seed} | margin {margin} | donors {n_donors} | cells {n_cells}")
print("test study fractions:", test_by_study)

# HARD STOP: match the frozen reference exactly (standing split-verification rule)
assert best_seed == 32,  f"seed {best_seed} != reference 32 -- split drifted, do NOT proceed"
assert margin == 10,     f"margin {margin} != reference 10 -- split drifted"
assert n_donors == {"train": 101, "val": 22, "test": 22}, f"donor counts {n_donors} != reference"
assert n_cells.get("train") == 94963 and n_cells.get("val") == 23824 and n_cells.get("test") == 23801, \
    f"cell counts {n_cells} != reference 94963/23824/23801"
print("split verification PASSED -- matches the frozen reference")

# --- per-study TRAIN cell counts: the denominator the epoch-matched budget (5a) is derived from ---
per_study_train = (glia.obs[glia.obs["split"] == "train"]
                   .groupby("study_id", observed=True).size().reindex(STUDIES).to_dict())
per_study_val   = (glia.obs[glia.obs["split"] == "val"]
                   .groupby("study_id", observed=True).size().reindex(STUDIES).to_dict())
print("per-study TRAIN cells:", {k: int(v) for k, v in per_study_train.items()})
print("per-study VAL   cells:", {k: int(v) for k, v in per_study_val.items()})
assert int(sum(per_study_train.values())) == 94963, "per-study train counts do not sum to the frozen 94,963"

# --- SMOKE subsample: AFTER the split is assigned, BEFORE tokenization (the dominant cost) ---
if SMOKE:
    keep = (glia.obs.groupby(["lineage", "split", "substate"], observed=True)
            .apply(lambda g: g.sample(min(len(g), SMOKE_N_PER_GROUP), random_state=SEED))
            .index.get_level_values(-1))
    glia = glia[glia.obs.index.isin(keep)].copy()
    print(f"[SMOKE] subsampled to {glia.n_obs} cells | split:", glia.obs["split"].value_counts().to_dict(),
          "| by study:", glia.obs["study_id"].value_counts().to_dict())

## 4 — Tokenize the substrate (once; all studies share one encoding)

### 4a — Map gene symbols to Ensembl IDs, gate on APOE vocabulary

In [ ]:
from geneformer import ENSEMBL_DICTIONARY_FILE, TOKEN_DICTIONARY_FILE
import pickle

with open(ENSEMBL_DICTIONARY_FILE, "rb") as f:
    symbol_to_ensembl = pickle.load(f)     # gene SYMBOL -> Ensembl ID
with open(TOKEN_DICTIONARY_FILE, "rb") as f:
    token_dictionary = pickle.load(f)      # Ensembl ID -> integer token

glia.var["ensembl_id"] = [symbol_to_ensembl.get(s) for s in glia.var_names]
mapped   = glia.var["ensembl_id"].notna()
in_vocab = glia.var["ensembl_id"].map(lambda e: (e in token_dictionary) if e is not None else False)
n_total, n_mapped, n_vocab = glia.n_vars, int(mapped.sum()), int(in_vocab.sum())
print(f"gene panel: {n_total} | mapped {n_mapped} ({n_mapped/n_total:.1%}) | in-vocab {n_vocab} ({n_vocab/n_total:.1%})")

NICHE_CRITICAL_GENES = ["APOE", "TREM2", "MS4A6A", "CLU", "GFAP", "AQP4", "AIF1", "CSF1R"]
panel = set(glia.var_names)
niche_status = {}
for g in NICHE_CRITICAL_GENES:
    e = symbol_to_ensembl.get(g) if g in panel else None
    niche_status[g] = {"in_panel": g in panel, "ensembl": e,
                       "in_vocab": bool(e in token_dictionary) if e is not None else False}
for g, s in niche_status.items():
    flag = "OK" if s["in_vocab"] else ("WARN" if s["in_panel"] else "ABSENT")
    print(f"  {g:8s} panel={str(s['in_panel']):5} vocab={str(s['in_vocab']):5}  [{flag}]")

# Pre-registered gate: APOE out-of-vocab makes eval #2 impossible for Geneformer -> HARD FAIL.
assert niche_status["APOE"]["in_vocab"], (
    "APOE not in Geneformer vocabulary -- eval #2 impossible. Pre-registered hard fail.")
niche_warnings = [g for g, s in niche_status.items() if not s["in_vocab"]]
VOCAB_AUDIT = {"n_genes_panel": n_total, "n_in_vocab": n_vocab,
               "frac_in_vocab": round(n_vocab / n_total, 4), "niche_warnings": niche_warnings,
               "apoe_hard_fail_gate": "passed"}

### 4b — Tokenize the full glia object (carries `split` + `study_id` so 5b can slice per study)

In [ ]:
from geneformer import TranscriptomeTokenizer

glia.obs["n_counts"] = np.asarray(glia.X.sum(axis=1)).ravel()
assert (glia.obs["n_counts"] > 0).all(), "cells with zero counts present -- should have been QC'd upstream"

TOK_IN_DIR, TOK_OUT_DIR = "/content/gf_token_in", "/content/gf_token_out"
os.makedirs(TOK_IN_DIR, exist_ok=True); os.makedirs(TOK_OUT_DIR, exist_ok=True)
glia.write_h5ad(os.path.join(TOK_IN_DIR, "glia.h5ad"))

# split + study_id + labels carried into every tokenized cell; cell_index is the realignment key.
CUSTOM_ATTRS = {c: c for c in
                ["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]}

# Upstream Geneformer tokenizer indexes var by integer POSITIONS while var.index holds symbols/Ensembl
# IDs; current pandas no longer falls back to positional indexing, so reset the post-sum_ensembl_ids
# var index to a RangeIndex (restores the behaviour the pinned commit's code assumes). Same patch as
# colab_09/11/12 -- keeps encoding identical to the zero-shot baseline.
import geneformer.tokenizer as _gftok
_orig_sum_ensembl_ids = _gftok.sum_ensembl_ids
def _sum_ensembl_ids_rangeindex_patch(*args, **kwargs):
    result = _orig_sum_ensembl_ids(*args, **kwargs)
    result.var.index = pd.RangeIndex(result.n_vars)
    return result
_gftok.sum_ensembl_ids = _sum_ensembl_ids_rangeindex_patch

tk = TranscriptomeTokenizer(CUSTOM_ATTRS, nproc=4)
tk.tokenize_data(TOK_IN_DIR, TOK_OUT_DIR, "glia_cpt", file_format="h5ad")
TOKENIZED_DATASET = os.path.join(TOK_OUT_DIR, "glia_cpt.dataset")
print("tokenized ->", TOKENIZED_DATASET)

## 5 — Per-study continued pretraining with LoRA

### 5a — Training budget + the `run_study()` routine

**Training-duration decision (per-study epoch matching).** The aggregated run (colab_11) trained
2000 steps at effective batch 32 over 94,963 train cells = **0.674 epochs**. Each per-study run here
is given the number of steps that reproduces *that same per-cell exposure* on its own train slice:

```
steps_S = round(TARGET_EPOCHS × n_train_S / effective_batch),   TARGET_EPOCHS = 0.674
```

Two consequences, both intended: (i) every study is trained to the same intensity per cell, so the
only thing separating this regime from aggregated is that gradients come from **one study at a time**
rather than a shuffled mixture; (ii) because the three train slices partition the 94,963 train cells,
the steps **sum to ≈ 2000** — the per-study regime spends the *same total gradient budget* as the
aggregated run, just partitioned by study.

The honest cost: **Haney2024** is the thinnest study, so its derived step count is small and its
adapter may barely move (a near-no-op checkpoint). That is a real property of a small-study CPT run,
not an artifact — it is printed and flagged, never floored up. *(Alternatives considered and not taken:
a fixed 2000 steps per study, which would overtrain the thin studies and confound regime with
intensity; or 2000 steps each at 3× total budget, which answers a different question.)*

LoRA config, masking objective, LR and collator are **identical to the aggregated v2 run** — the
adapter targets `["query","key","value","dense"]`, r=8, α=16, MLM prob 0.15. The base weights and the
embedding-tied MLM head stay frozen. Each study reloads a **fresh base** so no adapter is contaminated
by another study's training.

In [ ]:
import pickle, torch
from datasets import load_from_disk
from transformers import BertForMaskedLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
from geneformer import TOKEN_DICTIONARY_FILE, EmbExtractor
from geneformer.pretrainer import GeneformerPreCollator

# --- config, identical to the aggregated v2 run except max_steps (derived per study, see 5a) ---
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
LORA_TARGETS   = ["query", "key", "value", "dense"]
MLM_PROB       = 0.15
LEARNING_RATE  = 5e-4
PER_DEV_BATCH  = 8
GRAD_ACCUM     = 4
EFF_BATCH      = PER_DEV_BATCH * GRAD_ACCUM     # 32
WARMUP_RATIO   = 0.05
EVAL_STEPS_CAP = 250

# epoch target inherited from the aggregated run so per-cell exposure matches (5a).
AGG_STEPS, AGG_TRAIN_CELLS = 2000, 94963
TARGET_EPOCHS = AGG_STEPS * EFF_BATCH / AGG_TRAIN_CELLS      # 0.674...
print(f"TARGET_EPOCHS (from aggregated run) = {TARGET_EPOCHS:.4f}")

SLUG = {"SEA-AD": "seaad", "Li2025": "li2025", "Haney2024": "haney2024"}
LABEL_COLS = ["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]

with open(TOKEN_DICTIONARY_FILE, "rb") as f:
    token_dict = pickle.load(f)
assert "<mask>" in token_dict and "<pad>" in token_dict, "token dictionary missing <mask>/<pad>"
precollator = GeneformerPreCollator(token_dictionary=token_dict)
collator = DataCollatorForLanguageModeling(tokenizer=precollator, mlm=True, mlm_probability=MLM_PROB)

MODEL_DIR = os.path.join(GENEFORMER_REPO, "Geneformer-V2-104M")
assert os.path.exists(MODEL_DIR), f"model dir not found: {MODEL_DIR}"

full_ds = load_from_disk(TOKENIZED_DATASET)

# --- zero-shot baseline (colab_09), aligned to this substrate's cell_index, for detector #1 ---
ZEROSHOT_PATH = os.path.join(DRIVE_ROOT, "geneformer", "glia_geneformer_zeroshot.h5ad")
assert os.path.exists(ZEROSHOT_PATH), f"missing colab_09 baseline {ZEROSHOT_PATH}"
_zs = sc.read_h5ad(ZEROSHOT_PATH)
_zs_df = pd.DataFrame(_zs.X, index=_zs.obs["cell_index"].values).reindex(glia.obs["cell_index"].values)
assert _zs_df.notna().all().all(), "zero-shot baseline rows missing after cell_index alignment"
X_ZS = _zs_df.to_numpy(dtype=np.float32)
del _zs, _zs_df; gc.collect()

# detector #1 reference anchors (recalibrated 2026-07-16; docs/EVALUATION_CONTRACT.md)
NOISE_FLOOR        = 0.0000     # diag_colab_11_cpt_inert: base-vs-zeroshot rerun drift
SUBSTATE_REF_MICRO = 0.0365     # within-donor pairwise homeostatic vs activated (36 donors)
SUBSTATE_REF_ASTRO = 0.0442     # within-donor pairwise resting vs reactive (78 donors)

def _per_cell_cosine(A, B):
    num = (A * B).sum(1)
    den = np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1) + 1e-12
    return num / den

def _derive_max_steps(n_train):
    if SMOKE:
        return SMOKE_MAX_STEPS
    return max(1, round(TARGET_EPOCHS * n_train / EFF_BATCH))

def run_study(study):
    """Train one LoRA adapter on `study`'s train slice, re-embed the full substrate (L-1),
    run detector #1, save adapter + embedding, and return a result dict. Fresh base each call."""
    slug = SLUG[study]
    train_ds = full_ds.filter(lambda ex: ex["split"] == "train" and ex["study_id"] == study)
    val_ds   = full_ds.filter(lambda ex: ex["split"] == "val"   and ex["study_id"] == study)
    n_train, n_val = len(train_ds), len(val_ds)
    if n_train == 0:
        print(f"[skip] {study}: 0 train cells in this substrate"); return None
    steps      = _derive_max_steps(n_train)
    eval_steps = max(1, min(EVAL_STEPS_CAP, steps // 4)) if steps >= 4 else steps
    epochs     = steps * EFF_BATCH / n_train
    do_eval    = n_val > 0     # a thin study (esp. under SMOKE) can have 0 val cells -> disable eval
    print(f"\n=== {study} ===  train {n_train} | val {n_val} | max_steps {steps} "
          f"(~{epochs:.2f} epochs) | eval_steps {eval_steps if do_eval else 'n/a'}")
    if not do_eval:
        print(f"  NOTE: {study} has 0 val cells -- masked-LM eval disabled for this run.")
    if not SMOKE and steps < 200:
        print(f"  WARNING: {study} derives only {steps} steps -- adapter may barely move "
              "(thin-study near-no-op is a real result, not an error; reported, not floored).")

    base = BertForMaskedLM.from_pretrained(MODEL_DIR)
    lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                          target_modules=LORA_TARGETS, bias="none", task_type=TaskType.FEATURE_EXTRACTION)
    model = get_peft_model(base, lora_cfg)
    model.enable_input_require_grads()      # required for gradient checkpointing with a frozen base

    targs = TrainingArguments(
        output_dir=f"/content/gf_cpt_{slug}", overwrite_output_dir=True,
        max_steps=steps, per_device_train_batch_size=PER_DEV_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM, gradient_checkpointing=True,
        learning_rate=LEARNING_RATE, warmup_ratio=WARMUP_RATIO,
        eval_strategy="steps" if do_eval else "no", eval_steps=eval_steps if do_eval else None,
        logging_steps=max(1, steps // 10), save_strategy="no", report_to="none",
        bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
        seed=SEED, dataloader_num_workers=2)
    trainer = Trainer(model=model, args=targs,
                      train_dataset=train_ds.select_columns(["input_ids"]),
                      eval_dataset=val_ds.select_columns(["input_ids"]) if do_eval else None,
                      data_collator=collator)
    tres = trainer.train()
    train_loss = round(float(tres.training_loss), 4)
    print(f"  {study} train loss: {train_loss}")

    adapter_dir = os.path.join(DRIVE_ROOT, "geneformer", f"cpt_per_study_{slug}_seed0_adapter{SUFFIX}")
    os.makedirs(adapter_dir, exist_ok=True); model.save_pretrained(adapter_dir)

    # merge LoRA into the base, re-embed the FULL substrate at L-1 (pipeline readout)
    merged_dir = f"/content/gf_merged_{slug}"
    merged = model.merge_and_unload(); merged.save_pretrained(merged_dir)
    base.config.to_json_file(os.path.join(merged_dir, "config.json"))
    ee = EmbExtractor(model_type="Pretrained", num_classes=0, emb_mode="cell",
                      max_ncells=None, emb_layer=-1, emb_label=LABEL_COLS,
                      forward_batch_size=64, nproc=4)   # glia lengths known-safe at fbs=64 (colab_09/11/12)
    emb_dir = f"/content/gf_emb_{slug}"; os.makedirs(emb_dir, exist_ok=True)
    emb_df = ee.extract_embs(merged_dir, TOKENIZED_DATASET, emb_dir, f"glia_cpt_{slug}")
    emb_cols = [c for c in emb_df.columns if c not in LABEL_COLS]
    emb_df = emb_df.set_index("cell_index").reindex(glia.obs["cell_index"].values)
    assert emb_df[emb_cols].notna().all().all(), f"{study}: embedding rows missing after realignment"
    X_cpt = emb_df[emb_cols].to_numpy(dtype=np.float32)

    # detector #1: drift vs zero-shot, gated on held-out TEST
    cos = _per_cell_cosine(X_ZS, X_cpt)
    drift_all  = 1.0 - float(np.median(cos))
    test_mask  = (glia.obs["split"] == "test").values
    drift_test = 1.0 - float(np.median(cos[test_mask]))
    inert = np.isclose(drift_test, NOISE_FLOOR, atol=1e-4)
    det = {"drift_all": drift_all, "drift_test": drift_test, "noise_floor": NOISE_FLOOR,
           "gate": "held-out test", "inert": bool(inert),
           "pct_of_substate_ref_micro": None if inert else round(drift_test / SUBSTATE_REF_MICRO * 100, 1),
           "pct_of_substate_ref_astro": None if inert else round(drift_test / SUBSTATE_REF_ASTRO * 100, 1)}
    print(f"  detector #1 drift_test {drift_test:.4f} | {'INERT' if inert else 'REAL (above noise floor)'}")

    emb_ad = ad.AnnData(X=X_cpt, obs=glia.obs[LABEL_COLS].copy())
    emb_path = os.path.join(DRIVE_ROOT, "geneformer",
                            f"glia_geneformer_cpt_per_study_{slug}_seed0{SUFFIX}.h5ad")
    emb_ad.write_h5ad(emb_path)
    print(f"  saved adapter -> {os.path.relpath(adapter_dir, DRIVE_ROOT)} | emb -> {os.path.basename(emb_path)}")

    del base, model, merged, trainer, emb_df, X_cpt, emb_ad; gc.collect(); torch.cuda.empty_cache()
    return {"study": study, "slug": slug, "n_train": int(n_train), "n_val": int(n_val),
            "max_steps": int(steps), "epochs": round(epochs, 3), "train_loss": train_loss,
            "detector_1": det,
            "adapter_file": os.path.relpath(adapter_dir, DRIVE_ROOT),
            "embedding_file": os.path.relpath(emb_path, DRIVE_ROOT)}

### 5b — Run the three per-study CPT checkpoints

In [ ]:
results = {}
for study in STUDIES:
    r = run_study(study)
    if r is not None:
        results[study] = r
assert results, "no per-study checkpoints were produced -- every study had 0 train cells"

print("\n=== per-study CPT summary ===")
for s, r in results.items():
    print(f"{s:12} steps {r['max_steps']:5} (~{r['epochs']:.2f} ep) | train_loss {r['train_loss']:.4f} "
          f"| drift_test {r['detector_1']['drift_test']:.4f} "
          f"| {'INERT' if r['detector_1']['inert'] else 'REAL'}")

## 6 — Detector #1 across the three checkpoints

### 6a — Drift table vs the zero-shot baseline and the aggregated-run reference

In [ ]:
AGG_DRIFT_TEST = 0.005062   # aggregated v2 seed0 (audit_report.json geneformer_cpt_aggregated_v2)

rows = []
for s, r in results.items():
    d = r["detector_1"]
    rows.append({"study": s, "n_train": r["n_train"], "max_steps": r["max_steps"],
                 "epochs": r["epochs"], "train_loss": r["train_loss"],
                 "drift_test": round(d["drift_test"], 5),
                 "vs_aggregated": round(d["drift_test"] / AGG_DRIFT_TEST, 2),
                 "pct_ref_micro": d["pct_of_substate_ref_micro"],
                 "pct_ref_astro": d["pct_of_substate_ref_astro"],
                 "verdict": "INERT" if d["inert"] else "REAL"})
det_table = pd.DataFrame(rows).set_index("study")
print(det_table.to_string())
print(f"\naggregated v2 reference drift_test = {AGG_DRIFT_TEST:.5f}")
print("NOTE: detector #1 'REAL' only licenses proceeding to evals -- it is not itself a win. "
      "Any INERT checkpoint means that study's CPT did not move the embedding above the noise floor.")

## 7 — Save + handoff

### 7a — Append the audit trace, print the commit commands

In [ ]:
import shlex

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

if SMOKE:
    print("[SMOKE] plumbing run -- NOT writing audit_report.json")
else:
    report["geneformer_cpt_per_study"] = {
        "status": "computed", "date": TODAY, "regime": "per_study", "seed": SEED,
        "model_dir": os.path.basename(MODEL_DIR), "geneformer_commit": GENEFORMER_COMMIT,
        "n_cells": int(glia.n_obs), "emb_dim": 768,
        "budget_rule": "epoch-matched to aggregated run",
        "target_epochs": round(TARGET_EPOCHS, 4),
        "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "targets": LORA_TARGETS},
        "train_common": {"grad_accum": GRAD_ACCUM, "batch": PER_DEV_BATCH, "eff_batch": EFF_BATCH,
                         "lr": LEARNING_RATE, "mlm_prob": MLM_PROB},
        "per_study": {s: {k: r[k] for k in
                          ("slug", "n_train", "n_val", "max_steps", "epochs", "train_loss",
                           "detector_1", "adapter_file", "embedding_file")}
                      for s, r in results.items()},
        "donor_split_file": "outputs/donor_split.json",
        "compare_to": "geneformer_cpt_aggregated_v2",
    }
    with open(AUDIT_REPORT_PATH, "w") as f:
        json.dump(report, f, indent=2)
    print("audit trace appended ->", AUDIT_REPORT_PATH)

rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH)]
print("\n=== Commit + push (from WSL -- Colab has no git creds) ===")
print("  git add " + " ".join(shlex.quote(r) for r in rel))
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git commit -m "
      "'colab_14: Geneformer CPT (per-study regime, epoch-matched) + detector #1'")
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git push")

### Carried forward

Three per-study LoRA adapters + three full-substrate L−1 embeddings on Drive
(`geneformer/cpt_per_study_{seaad,li2025,haney2024}_seed0_adapter` and the matching
`glia_geneformer_cpt_per_study_*_seed0.h5ad`), plus a `geneformer_cpt_per_study` audit block.

**Next:** evals #1/#2 (substate probe, APOE) and #3 (forgetting) on the three checkpoints — a
downstream eval notebook that reuses the colab_12/13 machinery, reporting per-study mean ± SD against
the aggregated regime and the zero-shot baseline.